# VoiceDiary AI — Live Cloud GPU Platform
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

High-Performance Cloud GPU inference pipeline (`large-v3-turbo` default + 6 Model Hub + `ECAPA-TDNN` + `Gemini 2.5 Flash`).

---
### ⚡ Quick Start:
1. In the menu bar above, click **Runtime → Run all** (or press `Ctrl + F9`).
2. The interactive VoiceDiary application will render below in seconds!

In [ ]:
# 1. Install GPU Acceleration Libraries
!pip install -q --no-cache-dir faster-whisper speechbrain gradio soundfile torchaudio


In [ ]:
# 2. Launch Full-Fidelity VoiceDiary Web Platform (Complete Desktop Feature Parity)
import os, time, tempfile, json, re, urllib.request
import numpy as np
import soundfile as sf
import gradio as gr
import torch
import torchaudio
from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

# Hardware Acceleration Telemetry
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU Multi-Core (AVX2)'
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_dtype = 'float16' if device_type == 'cuda' else 'int8'
print(f'⚡ GPU Acceleration Active: {gpu_name} ({compute_dtype})')

# Global Model Hub Cache (Large-v3-Turbo Default)
loaded_models = {}

def get_whisper_model(model_name="large-v3-turbo"):
    if model_name not in loaded_models:
        print(f"Loading Whisper {model_name} onto GPU Tensor Cores...")
        os.makedirs("/content/models/whisper", exist_ok=True)
        loaded_models[model_name] = WhisperModel(
            model_name,
            device=device_type,
            compute_type=compute_dtype,
            num_workers=2,
            download_root="/content/models/whisper"
        )
    return loaded_models[model_name]

# Pre-load Large-v3-Turbo by default
print("Pre-warming OpenAI Whisper Large-v3-Turbo (809M)...")
default_model = get_whisper_model("large-v3-turbo")
print("✓ Large-v3-Turbo is hot in GPU VRAM.")

# Pre-load ECAPA-TDNN Diarizer
print("Pre-warming SpeechBrain ECAPA-TDNN Diarizer...")
os.makedirs("/content/models/ecapa", exist_ok=True)
embedder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/models/ecapa",
    run_opts={"device": device_type}
)
print("✓ ECAPA-TDNN Diarizer is hot in GPU VRAM.")

# Roman Urdu Transliteration Dictionary
URDU_TO_ROMAN_MAP = {
    'آپ': 'aap', 'کیسے': 'kaisay', 'ہیں': 'hain', 'کیا': 'kya', 'کر': 'kar', 'رہے': 'rahay', 'ہو': 'ho',
    'میں': 'main', 'ہوں': 'hoon', 'یہ': 'yeh', 'وہ': 'woh', 'نہیں': 'nahi', 'ٹھیک': 'theek', 'شکریہ': 'shukriya',
    'سلام': 'salam', 'علیکم': 'alaikum', 'سبق': 'sabaq', 'استاد': 'ustaad', 'کلاس': 'class', 'لیکچر': 'lecture',
    'سمجھ': 'samajh', 'آیا': 'aaya', 'آئی': 'aayi', 'پوچھنا': 'poochna', 'سوال': 'sawaal', 'جواب': 'jawab',
    'نوٹس': 'notes', 'کام': 'kaam', 'بہت': 'bohot', 'اچھا': 'acha', 'طالب': 'talib', 'علم': 'ilm'
}

def to_roman_urdu(text):
    words = text.split()
    out = []
    for w in words:
        clean_w = re.sub(r'[ً-ٰٟ]', '', w)
        out.append(URDU_TO_ROMAN_MAP.get(clean_w, w))
    return " ".join(out)

# Exact Desktop Midnight & Indigo CSS (1:1 with local desktop app)
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Fira+Code:wght@500;600&family=Noto+Nastaliq+Urdu:wght@400;700&display=swap');

:root {
    --bg-base: #080C14 !important;
    --bg-surface: #0F172A !important;
    --bg-surface-elevated: #1E293B !important;
    --accent-primary: #6366F1 !important;
    --accent-gradient: linear-gradient(135deg, #6366F1 0%, #8B5CF6 100%) !important;
    --text-primary: #F8FAFC !important;
    --text-secondary: #94A3B8 !important;
    --text-muted: #64748B !important;
}

body, .gradio-container {
    background-color: #080C14 !important;
    background-image: 
        radial-gradient(ellipse 80% 50% at 50% -20%, rgba(99, 102, 241, 0.15), transparent),
        radial-gradient(ellipse 60% 40% at 90% 90%, rgba(139, 92, 246, 0.08), transparent) !important;
    font-family: 'Plus Jakarta Sans', -apple-system, sans-serif !important;
    color: #F8FAFC !important;
    max-width: 1400px !important;
    margin: 0 auto !important;
}

/* Header */
.app-header {
    height: 64px;
    background: rgba(15, 23, 42, 0.75);
    backdrop-filter: blur(20px);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 16px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 0 20px;
    margin-bottom: 16px;
}

.header-left {
    display: flex;
    align-items: center;
    gap: 12px;
}

.brand-logo-circle {
    width: 38px;
    height: 38px;
    background: linear-gradient(135deg, #6366F1, #8B5CF6);
    border-radius: 10px;
    display: flex;
    align-items: center;
    justify-content: center;
    color: #FFFFFF;
    box-shadow: 0 0 15px rgba(99, 102, 241, 0.35);
}

.app-title {
    font-size: 17px;
    font-weight: 800;
    color: #FFFFFF;
    letter-spacing: -0.02em;
    margin: 0;
}

.app-subtitle {
    font-size: 11px;
    color: #94A3B8;
    font-weight: 500;
}

.header-right {
    display: flex;
    align-items: center;
    gap: 10px;
}

.pill-badge {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    background: rgba(16, 185, 129, 0.1);
    border: 1px solid rgba(16, 185, 129, 0.25);
    padding: 6px 14px;
    border-radius: 9999px;
    font-size: 11px;
    font-weight: 700;
    color: #10B981;
    font-family: 'Fira Code', monospace;
}

/* Panels */
.panel-box {
    background: rgba(15, 23, 42, 0.75) !important;
    border: 1px solid rgba(255, 255, 255, 0.08) !important;
    border-radius: 16px !important;
    padding: 18px !important;
    margin-bottom: 14px !important;
}

.panel-header-title {
    font-size: 11px;
    font-weight: 800;
    color: #94A3B8;
    letter-spacing: 0.06em;
    margin-bottom: 14px;
    display: flex;
    justify-content: space-between;
    align-items: center;
}

.count-tag {
    background: #1E293B;
    color: #6366F1;
    font-size: 11px;
    font-weight: 800;
    padding: 2px 8px;
    border-radius: 9999px;
}

.spk-card {
    background: rgba(255, 255, 255, 0.02);
    border: 1px solid rgba(255, 255, 255, 0.06);
    border-radius: 12px;
    padding: 10px 14px;
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 8px;
}

.spk-avatar {
    width: 34px;
    height: 34px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 800;
    font-size: 13px;
    color: #FFFFFF;
}

.transcript-viewport {
    background: #080C14;
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 14px;
    padding: 20px;
    min-height: 480px;
    max-height: 560px;
    overflow-y: auto;
}

.btn-primary-action {
    background: linear-gradient(135deg, #6366F1 0%, #8B5CF6 100%) !important;
    color: #FFFFFF !important;
    font-weight: 700 !important;
    font-size: 14px !important;
    border: none !important;
    border-radius: 12px !important;
    padding: 12px 20px !important;
    box-shadow: 0 0 20px rgba(99, 102, 241, 0.3) !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
    width: 100% !important;
}
.btn-primary-action:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 0 28px rgba(99, 102, 241, 0.5) !important;
}
"""

def fast_load_audio_16k(audio_path):
    try:
        waveform, sr = torchaudio.load(audio_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            waveform = resampler(waveform)
        audio_np = waveform.squeeze().numpy().astype(np.float32)
        return audio_np, len(audio_np) / 16000.0
    except Exception:
        data, sr = sf.read(audio_path)
        if data.ndim > 1:
            data = data.mean(axis=1)
        data = data.astype(np.float32)
        if sr != 16000:
            num_samples = int(len(data) * 16000 / sr)
            data = np.interp(np.linspace(0, len(data), num_samples, endpoint=False), np.arange(len(data)), data).astype(np.float32)
        return data, len(data) / 16000.0

def run_pipeline(audio_path, model_choice, lang_choice, sim_threshold, vad_silence):
    if not audio_path or not os.path.exists(audio_path):
        return "<div style='color:#EF4444;padding:30px;text-align:center;'>Please record speech or upload an audio file.</div>", "<div style='color:#64748B;padding:20px;text-align:center;'>No speakers detected</div>", None, None, None, None, ""
    
    t0 = time.time()
    data, duration = fast_load_audio_16k(audio_path)
    
    # 1. Resolve Active Model from Hub
    model_map = {
        '⚡ Large-v3-Turbo (809M) - OpenAI SOTA': 'large-v3-turbo',
        '⚡ Whisper Base (74M) - Fast Live': 'base',
        '⚡ Whisper Tiny (39M) - Ultralight': 'tiny',
        '⚡ Whisper Small (244M) - High Accuracy': 'small',
        '⚡ Whisper Medium (769M) - Deep Precision': 'medium',
        '⚡ Distil-Whisper (756M) - English Fast': 'distil-large-v3'
    }
    model_key = model_map.get(model_choice, 'large-v3-turbo')
    active_model = get_whisper_model(model_key)
    
    # 2. Resolve Language Mode
    is_roman = (lang_choice == '🔤 Roman Urdu (Latin)')
    lang_map = {
        '🌐 Bilingual (Urdu + English)': None,
        '🇵🇰 Pure Urdu Script (اردو)': 'ur',
        '🇬🇧 English Only': 'en',
        '🔤 Roman Urdu (Latin)': 'ur'
    }
    target_lang = lang_map.get(lang_choice, None)
    
    # 3. Fast Whisper Transcription
    segments, info = active_model.transcribe(
        data,
        beam_size=1,
        best_of=1,
        temperature=0.0,
        language=target_lang,
        without_timestamps=False,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=int(vad_silence)),
    )
    
    speaker_profiles = {}
    speaker_colors = ['#6366F1', '#10B981', '#F59E0B', '#EC4899', '#06B6D4', '#8B5CF6', '#F97316']
    next_speaker_id = 1
    
    transcript_html = []
    plain_entries = []
    srt_entries = []
    json_entries = []
    
    thresh = float(sim_threshold) / 100.0
    seg_idx = 1
    
    for seg in segments:
        text = seg.text.strip()
        if not text:
            continue
            
        if is_roman:
            text = to_roman_urdu(text)
            
        start_s = seg.start
        end_s = seg.end
        seg_audio = data[int(start_s * 16000): int(end_s * 16000)]
        
        spk_id = 1
        if len(seg_audio) >= 8000:
            try:
                with torch.inference_mode():
                    waveform = torch.from_numpy(seg_audio).float().unsqueeze(0).to(device_type)
                    emb = embedder.encode_batch(waveform).squeeze().detach().cpu().numpy()
                    emb_norm = emb / (np.linalg.norm(emb) or 1.0)
                    
                best_id = None
                best_sim = -1.0
                for s_id, embs in speaker_profiles.items():
                    sims = [float(np.dot(emb_norm, e)) for e in embs]
                    max_s = max(sims) if sims else 0
                    if max_s > best_sim:
                        best_sim = max_s
                        best_id = s_id
                        
                if best_id and best_sim >= thresh:
                    spk_id = best_id
                    if len(speaker_profiles[spk_id]) < 50:
                        speaker_profiles[spk_id].append(emb_norm)
                else:
                    spk_id = next_speaker_id
                    speaker_profiles[spk_id] = [emb_norm]
                    next_speaker_id += 1
            except Exception:
                pass
                
        spk_color = speaker_colors[(spk_id - 1) % len(speaker_colors)]
        time_str = f"{int(start_s // 60):02d}:{int(start_s % 60):02d}"
        
        # SRT Formatting
        def format_srt_time(sec):
            hrs = int(sec // 3600)
            mins = int((sec % 3600) // 60)
            secs = int(sec % 60)
            ms = int((sec - int(sec)) * 1000)
            return f"{hrs:02d}:{mins:02d}:{secs:02d},{ms:03d}"
            
        srt_entries.append(f"{seg_idx}\n{format_srt_time(start_s)} --> {format_srt_time(end_s)}\n[Speaker {spk_id}]: {text}\n")
        seg_idx += 1
        
        # JSON Formatting
        json_entries.append({
            "speaker": f"Speaker {spk_id}",
            "speaker_id": spk_id,
            "start": round(start_s, 2),
            "end": round(end_s, 2),
            "time": time_str,
            "text": text
        })
        
        is_urdu = any('؀' <= char <= 'ۿ' for char in text)
        rtl_style = "direction: rtl; text-align: right; font-family: 'Noto Nastaliq Urdu', serif; font-size: 16px; line-height: 1.8;" if is_urdu else "font-size: 14px; line-height: 1.6;"
        
        node = f"""
        <div style="margin-bottom: 12px; padding: 12px 16px; border-left: 4px solid {spk_color}; background: rgba(255,255,255,0.02); border-radius: 10px; border: 1px solid rgba(255,255,255,0.04);">
            <div style="display:flex; align-items:center; gap:8px; margin-bottom:4px;">
                <span style="display:inline-block; width:8px; height:8px; border-radius:50%; background:{spk_color};"></span>
                <strong style="color:{spk_color}; font-size:13px;">Speaker {spk_id}</strong>
                <span style="color:#64748B; font-size:11px; font-family:'Fira Code', monospace;">[{time_str}]</span>
            </div>
            <div style="color:#F8FAFC; {rtl_style}">{text}</div>
        </div>
        """
        transcript_html.append(node)
        plain_entries.append(f"[{time_str}] Speaker {spk_id}: {text}")
        
    elapsed = time.time() - t0
    
    # 4. Render Desktop Sidebar Speaker Cards
    sidebar_html = []
    for s_id, embs in speaker_profiles.items():
        c = speaker_colors[(s_id - 1) % len(speaker_colors)]
        sidebar_html.append(f"""
        <div class="spk-card">
            <div class="spk-avatar" style="background:{c};">S{s_id}</div>
            <div>
                <div style="font-weight:700; font-size:13px; color:#FFFFFF;">Speaker {s_id}</div>
                <div style="font-size:11px; color:#94A3B8;">{len(embs)} voice print{'s' if len(embs) > 1 else ''}</div>
            </div>
        </div>
        """)
    if not sidebar_html:
        sidebar_html.append("<div style='color:#64748B;padding:16px;text-align:center;'>No speakers detected</div>")
        
    full_html = "".join(transcript_html)
    full_html += f"""
    <div style="margin-top:16px; padding-top:12px; border-top:1px solid rgba(255,255,255,0.08); font-size:12px; color:#94A3B8; display:flex; justify-content:space-between;">
        <span>AI Engine: {model_key} ({gpu_name})</span>
        <span>Processed {duration:.1f}s in {elapsed:.2f}s ({(duration/max(0.01, elapsed)):.1f}x real-time speed)</span>
    </div>
    """
    
    # 5. Generate All 4 Downloadable Formats
    with tempfile.NamedTemporaryFile(mode='w', suffix='.md', delete=False, encoding='utf-8') as f:
        f.write(f"# VoiceDiary Lecture Notes\n\n" + "\n\n".join(plain_entries))
        md_file = f.name
        
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as f:
        f.write("\n".join(plain_entries))
        txt_file = f.name

    with tempfile.NamedTemporaryFile(mode='w', suffix='.srt', delete=False, encoding='utf-8') as f:
        f.write("\n".join(srt_entries))
        srt_file = f.name

    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False, encoding='utf-8') as f:
        json.dump(json_entries, f, indent=2, ensure_ascii=False)
        json_file = f.name
        
    return full_html, "".join(sidebar_html), md_file, txt_file, srt_file, json_file, "\n".join(plain_entries)

def generate_gemini_summary(transcript_text, gemini_api_key):
    if not transcript_text or not transcript_text.strip():
        return "No transcript content available to summarize."
        
    api_key = gemini_api_key.strip() if gemini_api_key else os.environ.get('GEMINI_API_KEY', '')
    if not api_key:
        return "Please paste your Gemini API Key in the left settings panel to generate structured study summaries."
        
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    payload = {
        "contents": [{
            "parts": [{
                "text": f"""You are VoiceDiary AI, an expert academic classroom study summarizer.
Analyze the following diarized lecture transcript and produce a structured Markdown study summary:
1. Executive Lecture Overview
2. Core Technical Concepts Discussed
3. Key Takeaways & Exam Points
4. Q&A / Discussion Highlights

Transcript:
{transcript_text}"""
            }]
        }]
    }
    
    try:
        req = urllib.request.Request(
            url,
            data=json.dumps(payload).encode('utf-8'),
            headers={'Content-Type': 'application/json'}
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode('utf-8'))
            return data['candidates'][0]['content']['parts'][0]['text']
    except Exception as e:
        return f"Gemini API Error: {e}"

with gr.Blocks(title="VoiceDiary — Bilingual Lecture & Diarization Engine", css=custom_css, theme=gr.themes.Default(primary_hue="indigo", neutral_hue="slate")) as demo:
    transcript_state = gr.State("")
    
    # Exact Top Header Bar
    gr.HTML(f"""
    <header class="app-header">
        <div class="header-left">
            <div class="brand-logo-circle">
                <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2"><path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/><path d="M19 10v2a7 7 0 0 1-14 0v-2"/><line x1="12" y1="19" x2="12" y2="23"/><line x1="8" y1="23" x2="16" y2="23"/></svg>
            </div>
            <div>
                <h1 class="app-title">VoiceDiary</h1>
                <div class="app-subtitle">AI Bilingual Lecture & Diarization Engine</div>
            </div>
        </div>
        <div class="header-right">
            <div class="pill-badge">
                <span style="display:inline-block;width:6px;height:6px;border-radius:50%;background:#10B981;"></span>
                <span>GPU: {gpu_name} (Tensor Cores FP16)</span>
            </div>
        </div>
    </header>
    """)
    
    with gr.Row():
        # LEFT COLUMN: Speakers & Profiles + Settings (Desktop Sidebar Match)
        with gr.Column(scale=3):
            with gr.Group(elem_classes=["panel-box"]):
                gr.HTML("<div class='panel-header-title'><span>SPEAKERS & PROFILES</span><span class='count-tag'>LIVE</span></div>")
                sidebar_out = gr.HTML(value="<div style='color:#64748B;padding:24px 12px;text-align:center;'><svg width='32' height='32' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='0.4' style='margin:0 auto 8px auto;'><path d='M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z'/><path d='M19 10v2a7 7 0 0 1-14 0v-2'/></svg><p>No speakers detected</p><p style='font-size:11px;color:#64748B;'>Start recording to identify professor & students</p></div>")
                
            with gr.Group(elem_classes=["panel-box"]):
                gr.HTML("<div class='panel-header-title'><span>AI ENGINE & MODEL CONFIGURATION</span></div>")
                model_dropdown = gr.Dropdown(
                    choices=[
                        '⚡ Large-v3-Turbo (809M) - OpenAI SOTA',
                        '⚡ Whisper Base (74M) - Fast Live',
                        '⚡ Whisper Tiny (39M) - Ultralight',
                        '⚡ Whisper Small (244M) - High Accuracy',
                        '⚡ Whisper Medium (769M) - Deep Precision',
                        '⚡ Distil-Whisper (756M) - English Fast'
                    ],
                    value='⚡ Large-v3-Turbo (809M) - OpenAI SOTA',
                    label='Active Whisper Model'
                )
                lang_dropdown = gr.Dropdown(
                    choices=[
                        '🌐 Bilingual (Urdu + English)',
                        '🇵🇰 Pure Urdu Script (اردو)',
                        '🇬🇧 English Only',
                        '🔤 Roman Urdu (Latin)'
                    ],
                    value='🌐 Bilingual (Urdu + English)',
                    label='Language Output Mode'
                )
                thresh_slider = gr.Slider(minimum=20, maximum=70, value=32, step=1, label='Diarization Sensitivity (Cosine %)')
                vad_slider = gr.Slider(minimum=150, maximum=600, value=280, step=10, label='VAD Silence Gap (ms)')
                gemini_key_in = gr.Textbox(placeholder='Paste Gemini API Key here...', type='password', label='✨ Gemini AI Key (BYOK)')

        # RIGHT COLUMN: Audio Capture & Live Transcript
        with gr.Column(scale=7):
            with gr.Group(elem_classes=["panel-box"]):
                with gr.Tabs():
                    with gr.TabItem("🎙️ Live Classroom Lecture"):
                        audio_mic = gr.Audio(sources=["microphone"], type="filepath", label="Capture Classroom Speech")
                    with gr.TabItem("📁 Upload Audio File"):
                        audio_file = gr.Audio(sources=["upload"], type="filepath", label="Upload Lecture Audio File (.wav, .mp3, .m4a, .flac)")
                
                transcribe_btn = gr.Button("Transcribe & Diarize Lecture (GPU Accelerated)", elem_classes=["btn-primary-action"])
            
            with gr.Group(elem_classes=["panel-box"]):
                gr.HTML("<div class='panel-header-title'><span>CLASSROOM LECTURE TRANSCRIPT</span></div>")
                transcript_display = gr.HTML(
                    value="<div style='display:flex; flex-direction:column; align-items:center; justify-content:center; height:320px; color:#64748B; text-align:center;'><svg width='40' height='40' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='0.4' style='margin-bottom:12px;'><path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/></svg><p style='font-size:15px; font-weight:600; color:#94A3B8;'>Lecture transcript will stream here</p><p style='font-size:12px; color:#64748B;'>Click the button above to begin live GPU transcription</p></div>",
                    elem_classes=["transcript-viewport"]
                )
            
            # Export & Gemini Study Summary (All 4 Formats + AI)
            with gr.Group(elem_classes=["panel-box"]):
                gr.HTML("<div class='panel-header-title'><span>EXPORT NOTES & AI STUDY FLASHCARDS</span></div>")
                with gr.Row():
                    d_md = gr.File(label="Markdown Notes (.md)")
                    d_txt = gr.File(label="Plain Text (.txt)")
                with gr.Row():
                    d_srt = gr.File(label="Timed Subtitles (.srt)")
                    d_json = gr.File(label="Structured JSON (.json)")
                
                ai_sum_btn = gr.Button("✨ Generate AI Lecture Summary & Flashcards (Gemini 2.5 Flash)", elem_classes=["btn-primary-action"])
                summary_out = gr.Markdown(value="*AI summary and study flashcards will appear here after clicking above...*")

    # Wire event handlers
    transcribe_btn.click(
        fn=lambda m, f, mod, lang, th, vad: run_pipeline(m if m else f, mod, lang, th, vad),
        inputs=[audio_mic, audio_file, model_dropdown, lang_dropdown, thresh_slider, vad_slider],
        outputs=[transcript_display, sidebar_out, d_md, d_txt, d_srt, d_json, transcript_state]
    )
    
    ai_sum_btn.click(
        fn=generate_gemini_summary,
        inputs=[transcript_state, gemini_key_in],
        outputs=[summary_out]
    )

demo.queue(max_size=20).launch(share=True, inline=True, debug=False, show_error=True)
